INTRINCS
     -Usare delle estensioni del compilatore che espongono alcune
     istruzioni come funzioni. Problema: il codice non è portabile ad altre architetture.
     -Richiedono: #include <immintrin.h> e compilazione con -mavx
          ```c
          __m256 x = _mm256_loadu_ps(&a[i]);   // carica 8 float
          __m256 y = _mm256_loadu_ps(&b[i]);
          __m256 z = _mm256_add_ps(x, y);      // somma 8 float in parallelo
          _mm256_storeu_ps(&c[i], z);          // salva 8 float


In [ ]:
# include < stdio .h >
2 # include < stdlib .h >
3 # include < time .h >
4 # include < immintrin .h >
5
6 float * make_array ( int n)
7 {
8 float * v = ( float *) malloc (n * sizeof ( float ));
9 for ( int i = 0; i < n; i ++) {
10 v[i ] = ( float ) rand () / RAND_MAX ;
11 }
12 return v;
13 }
14
15 int main ( int argc , char * argv [])
16 {
17 const int n = 1 < <20;
18 float * a = make_array (n );
19 float * b = make_array (n );
20 float * c = make_array (n );
21
22 clock_t start = clock () ;
23 for ( int i = 0; i < n; i += 8) {
24 __m256 x = _mm256_loadu_ps (& a [i ]) ;
25 __m256 y = _mm256_loadu_ps (& b [i ]) ;
26 __m256 z = _mm256_add_ps (x , y);
27 _mm256_storeu_ps (& c[ i], z);
28 }
29 clock_t end = clock () ;
30 float ms = ( float ) ( end - start ) / ( CLOCKS_PER_SEC / 1000.0) ;
31 printf (" Somma con istruzioni vettoriali : %f ms\n", ms );
32
33 start = clock () ;
34 for ( int i = 0; i < n; i ++) {
35 c[i ] = a[ i] + b[ i ];
36 }
37 end = clock () ;
38 ms = ( float ) ( end - start ) / ( CLOCKS_PER_SEC / 1000.0) ;
39 printf (" Somma senza istruzioni vettoriali : %f ms\n", ms );
40
41 return 0;
42 }

vettoriali per GCC
    - fornisce estensioni vettoriali generiche
    -sintassi: 
        typedef int v4si __attribute__ ((vector_size (4 * sizeof(int))));

In [6]:
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

typedef int v8si __attribute__ ((vector_size (8 * sizeof(int))));

int sum_vector(int *vec, int len)
{
    int sum = 0;
    for (int i = 0; i < len; i++) {
        sum += vec[i];
    }
    return sum;
}

int sum_vector_simd(int *vec, int len)
{
    v8si sumv = {0, 0, 0, 0, 0, 0, 0, 0};
    int i;
    for (i = 0; i + 7 < len; i += 8) {
        v8si x = *(v8si *)&vec[i];
        sumv += x;
    }

    int sum = sumv[0] + sumv[1] + sumv[2] + sumv[3] + sumv[4] + sumv[5] + sumv[6] + sumv[7];
    for (; i < len; i++) {
        sum += vec[i];
    }
    return sum;
}

float benchmark_scalar_100(int *vec, int len, int *out_sum)
{
    const int runs = 100;
    clock_t start = clock();
    int s = 0;
    for (int r = 0; r < runs; r++) {
        s = sum_vector(vec, len);
    }
    clock_t end = clock();
    if (out_sum != NULL) {
        *out_sum = s;
    }
    return (float)(end - start) / (CLOCKS_PER_SEC / 1000.0f) / runs;
}

float benchmark_simd_100(int *vec, int len, int *out_sum)
{
    const int runs = 100;
    clock_t start = clock();
    int s = 0;
    for (int r = 0; r < runs; r++) {
        s = sum_vector_simd(vec, len);
    }
    clock_t end = clock();
    if (out_sum != NULL) {
        *out_sum = s;
    }
    return (float)(end - start) / (CLOCKS_PER_SEC / 1000.0f) / runs;
}

#define N 1000000

int main(int argc, char *argv[])
{
    int v[N];
    for (int i = 0; i < N; i++) {
        v[i] = rand() % 100;
    }

    int s = 0;
    int sv = 0;

    float ms_scalar = benchmark_scalar_100(v, N, &s);
    float ms_simd = benchmark_simd_100(v, N, &sv);

    printf("scalar avg (100 run): %f ms\n", ms_scalar);
    printf("simd avg (100 run): %f ms\n", ms_simd);
    printf("s = %d, sv = %d\n", s, sv);

    return 0;
}

scalar avg (100 run): 0.956080 ms
simd avg (100 run): 0.437960 ms
s = 49483647, sv = 49483647


Multithreading con OpenMP
    -api supporto programmazione parallerla in sistemi multi processore e memoria condivisa
    -viene usato dai thread
        -processori eseguono codice parallelo, accedono stessa memoria
    -tipologie: processori ma memoria non condivisa

    -struttura: thread princiaple parte secondari, distribuzione lavoro thread

    -utilizzo: importare omp.h, compilare con -fopenmp(su macos devi usare clang con llvm-openmp)
    -pragma #pragma omp parallel{ codice eseguito tutti thread}